# metal-graph in 5 minutes

[metal-graph](https://github.com/tabulai/metal-graph) runs graph analytics
(PageRank, batched personalized PageRank with top-k, BFS, k-hop, WCC) as
Metal kernels on Apple Silicon, with threaded CPU implementations behind the
same API and a per-operation planner.

This notebook tours the core API on a tiny graph: building from arbitrary
IDs, running every algorithm, checking results against NetworkX, and reading
the execution telemetry. Everything here runs in well under a second.

In [1]:
import numpy as np
import metal_graph as mg

print("metal-graph", mg.__version__, "| Metal GPU available:", mg.has_gpu())

metal-graph 0.1.0 | Metal GPU available: True


## Build a graph from string IDs

`from_edges` accepts int32/int64/str IDs and maps them to dense **user
indices** `0..V-1` (`np.unique` order). Every algorithm speaks user indices;
`G.external_ids` and `G.index_of` translate both ways.

In [2]:
src = np.array(["alice", "alice", "bob",  "carol", "dave", "erin",
                "frank", "grace", "carol"])
dst = np.array(["bob",   "carol", "carol", "dave",  "alice", "frank",
                "erin",  "erin",  "alice"])
G = mg.Graph.from_edges(src, dst, directed=True)

print("V =", G.num_vertices, "| E =", G.num_edges)
print("external_ids:", G.external_ids.tolist())
print("index_of(['carol', 'erin']):", G.index_of(["carol", "erin"]).tolist())

V = 7 | E = 9
external_ids: ['alice', 'bob', 'carol', 'dave', 'erin', 'frank', 'grace']
index_of(['carol', 'erin']): [2, 4]


## PageRank — NetworkX-compatible semantics

Same iteration, same dangling handling, same weighted normalization. The
suite validates this on 1,000+ golden tests; here is the spot check.

In [3]:
import networkx as nx

pr = np.asarray(mg.pagerank(G, alpha=0.85, tol=1e-10, max_iter=200))

nxg = nx.DiGraph()
nxg.add_nodes_from(G.external_ids)
nxg.add_edges_from(zip(src, dst))
nx_pr = nx.pagerank(nxg, alpha=0.85, tol=1e-10, max_iter=200)
nx_vec = np.array([nx_pr[e] for e in G.external_ids])

print("max |metal-graph - networkx| =", float(np.abs(pr - nx_vec).max()))
top = np.argsort(-pr)[:3]
print("top-3:", [(str(G.external_ids[i]), round(float(pr[i]), 4)) for i in top])

max |metal-graph - networkx| = 7.13313735856147e-09
top-3: [('erin', 0.2085), ('frank', 0.1986), ('alice', 0.1855)]


## BFS, k-hop and connected components

In [4]:
alice = int(G.index_of("alice"))
dist, parent = mg.bfs(G, sources=[alice], direction="out")
print("BFS depths from alice:",
      {str(G.external_ids[i]): int(d) for i, d in enumerate(dist) if d >= 0})

vs, es = mg.k_hop(G, seeds=[alice], k=2, direction="both")
print("2-hop neighborhood:", [str(G.external_ids[v]) for v in vs])
sub = mg.k_hop(G, seeds=[alice], k=2, direction="both", as_graph=True)
print("materialized subgraph: V =", sub.num_vertices, "E =", sub.num_edges)

comp = np.asarray(mg.experimental.wcc(G))
print("weakly connected components:", int(comp.max()) + 1)

BFS depths from alice: {'alice': 0, 'bob': 1, 'carol': 1, 'dave': 2}
2-hop neighborhood: ['alice', 'bob', 'carol', 'dave']
materialized subgraph: V = 4 E = 6
weakly connected components: 2


## The planner and honest telemetry

`mode="auto"` picks CPU or GPU per operation (GPU pays off from roughly a
million edges up — this toy graph stays on CPU). `last_run_info()` always
reports what actually executed; there is no silent fallback.

In [5]:
for mode in (["cpu", "gpu"] if mg.has_gpu() else ["cpu"]):
    mg.set_execution(mode)
    mg.pagerank(G, alpha=0.85, tol=1e-8, max_iter=100)
    print(f"mode={mode!r:6} ->", mg.last_run_info())
mg.set_execution("auto")

mode='cpu'  -> {'op': 'pagerank', 'path': 'cpu', 'iterations': 95, 'ms': 0.073208}
mode='gpu'  -> {'op': 'pagerank', 'path': 'gpu', 'iterations': 95, 'ms': 9.948125}


Next: [`02_batched_ppr_retrieval.ipynb`](02_batched_ppr_retrieval.ipynb)
shows the flagship batched-PPR API on an agent-retrieval workload, and
[`03_bfs_latency_planner.ipynb`](03_bfs_latency_planner.ipynb) shows the
microsecond BFS latency path on a 2M-edge graph.